# W04-B · Is the classifier any good?

**Your job.** A support team wants every unhappy review flagged, so they can reply to the customer. You have the Naïve
Bayes classifier from W04-A, trained on 13 reviews. Before anyone relies on it, find out how good it is.

To answer fairly, we test it on **10 new reviews** it did not learn from, the way an exam uses questions you haven't
seen before. For each one we know the real label.

| Task | What you do |
|---|---|
| 1 | Predict a label for each test review |
| 2 | Accuracy: the share it gets right |
| 3 | The do-nothing baseline: always say positive |
| 4 | The confusion table |
| 5 | Precision and recall for negative reviews |

Every ✍️ task checks itself.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w04/W04-B-is-it-any-good.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w04")

In [ ]:
# ▶ Setup: just run this cell. The training reviews and the classifier are the same as in W04-A.
from collections import Counter   # counting (see W04-A)

# The 13 reviews the classifier learns from, the same as in W04-A.
train = [
    ("great filter works well", "pos"),
    ("easy to install and works great", "pos"),
    ("love it great price", "pos"),
    ("works perfectly and fits well", "pos"),
    ("good value easy fit", "pos"),
    ("quiet and fast love it", "pos"),
    ("great product good price", "pos"),
    ("good quality and easy to use", "pos"),
    ("stopped working after a week", "neg"),
    ("leaks and broke fast", "neg"),
    ("bad fit not worth the money", "neg"),
    ("cheap and noisy not good", "neg"),
    ("does not work bad quality", "neg"),
]
# Each test review comes with its real label, so we can mark the classifier's answers.
test = [                  # 10 new reviews: the classifier never counts these
    ("works great and quiet", "pos"),
    ("easy install good price", "pos"),
    ("love it fits well", "pos"),
    ("great value works perfectly", "pos"),
    ("not bad at all", "pos"),
    ("very quiet fits well", "pos"),
    ("broke after a week", "neg"),
    ("noisy and leaks", "neg"),
    ("not great not good", "neg"),
    ("looks great but it leaks", "neg"),
]

# Training: count the positive reviews and the words of each pile, all in one loop
# (W04-A, Exercises 1 and 2). Only `train` is counted, never `test`.
n_pos = 0
pos_counts = Counter()
neg_counts = Counter()
for text, label in train:
    if label == "pos":
        n_pos = n_pos + 1
        pos_counts.update(text.split())
    else:
        neg_counts.update(text.split())
prior_pos = n_pos / len(train)              # P(pos): the share of positive reviews
prior_neg = 1 - prior_pos                   # P(neg): the rest
vocab = set(pos_counts) | set(neg_counts)   # every word the training reviews use
V = len(vocab)                              # how many different words

def p_word(word, counts):
    # W04-A, Exercise 3: add-one smoothing
    return (counts[word] + 1) / (sum(counts.values()) + V)   # (count + 1) / (words in the pile + V)

def score(review, prior, counts):
    # W04-A, Exercise 4: prior × every known word's probability
    s = prior
    for word in review.split():
        if word not in vocab:
            continue                         # never seen in training: skip it
        s = s * p_word(word, counts)
    return s

def classify(review):
    # W04-A, Exercise 5: the label with the bigger score, and the chance of positive
    pos_score = score(review, prior_pos, pos_counts)
    neg_score = score(review, prior_neg, neg_counts)
    if pos_score > neg_score:
        label = "pos"
    else:
        label = "neg"
    return label, pos_score / (pos_score + neg_score)

print(len(train), "training reviews,", len(test), "test reviews")

## Task 1 · Predict a label for each test review

### ✍️ Exercise 1: `predicted`

Make `predicted`: a list with the classifier's label for each test review, in order. `classify(text)` returns two
things; you only need the first, `classify(text)[0]`.

In [ ]:
predicted = []                            # an empty list; .append(x) adds x to its end
for text, label in test:                  # the classifier only gets the text, never the label
    ...   # ✍️ add the predicted label of this text to the list
# real: the real labels of the test reviews, in the same order as predicted.
real = []
for text, label in test:
    real.append(label)

# zip() walks through two lists side by side: each test pair together with its prediction.
for (text, label), guess in zip(test, predicted):
    mark = "✅" if guess == label else "❌"   # an if on one line: ✅ for a right guess, ❌ for a wrong one
    print(f"{mark} real {label}, said {guess}   {text}")
# Check: 10 predictions, and the first review, "works great and quiet", comes out positive.
assert len(predicted) == 10 and predicted[0] == "pos", "not quite"
print("✅")

## Task 2 · Accuracy

**Accuracy** = the number of correct predictions ÷ the number of predictions.

### ✍️ Exercise 2: `accuracy(real, predicted)`

Hint: `zip(real, predicted)` gives each real label with its prediction.

In [ ]:
def accuracy(real, predicted):
    correct = 0                         # the tally of right predictions
    for r, p in zip(real, predicted):   # r: a real label, p: the prediction for the same review
        ...   # ✍️ add 1 to correct if the prediction matches
    return ...   # ✍️
print("Naïve Bayes accuracy:", accuracy(real, predicted))
# Check on a tiny example: one right ("pos", "pos") and one wrong ("neg", "pos"), so 0.5.
assert accuracy(["pos", "neg"], ["pos", "pos"]) == 0.5, "not quite"
print("✅")

## Task 3 · The do-nothing baseline

Is 80% good? Compare it with a "classifier" that doesn't read the review at all and **always says positive**. Any real
classifier has to beat this **baseline**.

### ✍️ Exercise 3: `baseline`

Make `baseline`: a list that says `"pos"` once for every test review, then work out its accuracy.

In [ ]:
baseline = ...   # ✍️ hint: ["pos"] * 3 is ["pos", "pos", "pos"]
# The same accuracy function as before, now marking the always-positive guesses.
print("Always-positive accuracy:", accuracy(real, baseline))
assert baseline == ["pos"] * 10, "not quite"
print("✅")

> 📖 **Reading the output.** Doing nothing scores 60%, because 6 of the 10 test reviews are positive. Our classifier's
> 80% is better, but by 20 points, not 80. If 95% of a shop's reviews were positive, always saying positive would
> score 95%, and still catch **no** unhappy customer. Accuracy alone can hide that.

## Task 4 · The confusion table

The support team cares about negative reviews, so treat **negative** as the alarm, like *"WOLF!"*. Every test review
lands in one of four boxes:

| | said negative | said positive |
|---|---|---|
| **really negative** | **caught** | **missed** |
| **really positive** | **false alarm** | correct positive |

### ✍️ Exercise 4: `confusion(real, predicted)`

Return a `Counter` with the four boxes: `"caught"`, `"missed"`, `"false alarm"` and `"correct positive"`. Two are
filled in for you.

In [ ]:
def confusion(real, predicted):
    boxes = Counter()                   # one tally per box; a box nothing lands in stays at 0
    for r, p in zip(real, predicted):   # r: the real label, p: what the classifier said
        # if / elif / else: Python tries each test in order and runs only the first one that is true.
        if r == "neg" and p == "neg":   # and: both must be true
            boxes["caught"] += 1        # += 1 is short for: boxes["caught"] = boxes["caught"] + 1
        elif ...:   # ✍️ really negative, said positive
            boxes["missed"] += 1
        elif ...:   # ✍️ really positive, said negative
            boxes["false alarm"] += 1
        else:                           # the only case left: really positive, said positive
            boxes["correct positive"] += 1
    return boxes
nb = confusion(real, predicted)    # the four boxes for our classifier
base = confusion(real, baseline)   # the four boxes for always-positive
print("Naïve Bayes:    ", dict(nb))      # dict(...) prints the Counter as a plain dictionary
print("Always positive:", dict(base))
assert nb == Counter({"caught": 3, "missed": 1, "false alarm": 1, "correct positive": 5}), "not quite"
print("✅")

## Task 5 · Precision and recall

Two questions about the alarm:

- **Precision**: of the reviews it **flagged** negative, how many really were? caught ÷ (caught + false alarm).
  This is the wolf question: of all the *"WOLF!"* alarms, how many were real wolves?
- **Recall**: of the reviews that **really** are negative, how many did it flag? caught ÷ (caught + missed).

### ✍️ Exercise 5: `precision` and `recall`

Work both out from the Naïve Bayes boxes, `nb`.

In [ ]:
precision = ...   # ✍️ of the reviews it flagged negative, the share that really are
recall = ...      # ✍️ of the reviews that really are negative, the share it flagged
print(f"precision {precision:.0%}   recall {recall:.0%}")   # :.0% shows 0.75 as 75%
assert (precision, recall) == (0.75, 0.75), "not quite"
print("✅")
# Always-positive never says negative, so it catches none of them: recall 0.
# Its precision would be 0 ÷ 0, which has no answer.
print("Always positive: recall", base["caught"] / (base["caught"] + base["missed"]), "(precision can't be worked out: it never flags anything)")

## Look at the mistakes

Accuracy, precision and recall say **how often** the classifier is wrong. The mistakes themselves say **why**.
Run the cell: for each mistake it shows how each word pulled, P(word | pos) ÷ P(word | neg). Above 1 pulls towards
positive, below 1 towards negative, and `?` means the word was never seen in training.

In [ ]:
# ▶ Just run this cell.
for (text, label), guess in zip(test, predicted):
    if guess == label:
        continue                                      # a right answer: skip it, we only want the mistakes
    print(f"real {label}, said {guess}: {text!r}")    # !r prints the text with quotes around it
    for word in text.split():                         # how did each word pull?
        if word in vocab:
            # {word:<8} pads the word to 8 characters, so the numbers line up in a column.
            print(f"    {word:<8} {p_word(word, pos_counts) / p_word(word, neg_counts):.2f}")
        else:
            print(f"    {word:<8} ?")                 # never seen in training: no pull either way
    print()                                           # an empty line between mistakes

> 📖 **Reading the output.**
> - *"not bad at all"* is a happy review, but *not* and *bad* only ever appeared in negative reviews. The classifier
>   sees separate words, never the phrase *not bad*. That is the naïve assumption going wrong.
> - *"looks great but it leaks"* is unhappy, but *great* is the strongest positive clue there is, *it* leans positive
>   too (*love it*), and *looks* and *but* are unknown. *leaks* alone can't outweigh them plus a prior that favours
>   positive.

## For the class discussion

1. In plain words, what do *"precision 75%"* and *"recall 75%"* mean for the support team?
2. The team can only read 20 flagged reviews a day. Which matters more to them, precision or recall? And if a missed
   review could be about a safety problem, such as a heater that overheats?
3. Our classifier read 13 reviews. Name two things you would change before trusting it with a real shop's reviews.